In [54]:
import pandas as pd
import re
df = pd.read_csv('2026-03-30.csv', encoding='utf-8')

In [55]:
# 清洗全文

def clean_content(text):
    if not isinstance(text, str):
        return text
    
    # 删除 "NEW " 字段
    text = text.replace('NEW ', '')
    
    # 任务2：删除 "求人票No ... あとからチェックしたい方は 気になる" 之后的所有内容
    # 我们定位到 "気になる" 这个关键词并截断
    # 如果关键词非常固定，可以使用 split
    keyword = "あとからチェックしたい方は 気になる"
    if keyword in text:
        text = text.split(keyword)[0]
    
    # 额外清理：由于截断后末尾可能残留 "求人票No K..." 等信息，
    # 我们可以用正则表达式进一步精准定位到 "求人票No" 之前
    text = re.sub(r'求人票No\s+K\d{8}-\d{3}-\d{2}-\d{2}.*$', '', text, flags=re.DOTALL)
    
    return text.strip()

df['内容全文'] = df['内容全文'].apply(clean_content)




In [56]:
#清洗公司和position
def extract_advanced(text):
    if not isinstance(text, str):
        return None, None

    # 1. 扩充公司特征：加入“相互会社”并明确包含全角空格
    # \S* 表示匹配非空白字符（含全角字符）
    company_pattern = r"(\S*(?:株式会社|合同会社|有限会社|相互会社|（株）|\(株\))\S*)"
    
    # 2. 护城河标签：同样适配全角/半角空格
    stop_tags = r"(?:業界未経験|フレックス|年間休日|土曜出勤なし|正社員|想定年収|勤務地)"
    
    # 3. 核心正则表达式改动：
    # ^\s* : 容错开头可能存在的换行或不可见字符
    # (.*?) : 职位名
    # [ \s　]+ : 匹配半角、全角或Tab空格
    regex = rf"^\s*(.*?)[ \s　]+{company_pattern}(?:[ \s　]+{stop_tags}|$)"
    
    match = re.search(regex, text)
    
    if match:
        job_title = match.group(1).strip()
        company_name = match.group(2).strip()
        
        # 针对 NEC 等带括号的特殊处理：检查公司名后紧跟的括号
        extra_parentheses = re.search(r"^[（\(][^）\)]+[）\)]", text[match.end(2):])
        if extra_parentheses:
            company_name += extra_parentheses.group()
            
        return job_title, company_name
    
    # 保底方案：使用正则 split 兼容全角/半角
    parts = re.split(r'[\s　]+', text.strip())
    if len(parts) >= 2:
        for i, p in enumerate(parts):
            if any(x in p for x in ['株式会社', '合同会社', '有限会社', '相互会社', '（株）']):
                return " ".join(parts[:i]), parts[i]
                
    return text[:20], "未识别"
# 应用
df[['Position', '会社名']] = df['内容全文'].apply(lambda x: pd.Series(extract_advanced(x)))


In [57]:

# 提取其他信息
def extract_fields(text):
    # 这里的 text 已经是之前处理过的（删除了 NEW 和 尾部冗余）
    clean_text = text

    
    if not isinstance(clean_text, str) or clean_text.strip() == "":
        return pd.Series([''] * 9)

    # === 3. 提取在宅/Remote 关键词 (优化版) ===
    # 加入了 ハイブリッド、出社 等关键词，并捕捉前后 25 个字符以获取更完整的语境（如：原则出社、不可等）
    # 使用 re.IGNORECASE 以防万一有英文大写
    pattern = r'(.{0,5}(?:在宅|リモート|テレワーク|ハイブリッド|出社).{0,25})'
    remote_keywords = re.findall(pattern, clean_text)
    
    if remote_keywords:
        # 去重并合并
        remote_info = " || ".join(dict.fromkeys([k.strip() for k in remote_keywords]))
    else:
        remote_info = "无匹配"

    # === 4. 提取特定段落 ===
    def extract_between(source, start_str, end_str):
        start_idx = source.find(start_str)
        if start_idx == -1:
            return ""
        start_idx += len(start_str)
        end_idx = source.find(end_str, start_idx)
        if end_idx == -1:
            # 如果没找到结束标志，截取后面 500 字
            return source[start_idx:].strip()[:500] 
        return source[start_idx:end_idx].strip()

    job_content = extract_between(clean_text, "仕事の内容", "配属先情報")
    dept_info = extract_between(clean_text, "配属先情報", "必要な能力・経験")
    requirements = extract_between(clean_text, "必要な能力・経験", "勤務地") 
    salary = extract_between(clean_text, "給与", "就業時間")
    work_hours = extract_between(clean_text, "就業時間", "通勤手当")
    selection = extract_between(clean_text, "選考内容", "企業情報")
    
    # 提取员工人数
    emp_count_match = re.search(r'従業員数[\s　]*([0-9,]+)', clean_text)
    emp_count = emp_count_match.group(1) if emp_count_match else ""

    return pd.Series([
        emp_count, 
        remote_info, 
        salary,
        requirements, 
        job_content,  
        dept_info, 
        work_hours, 
        selection, 
        clean_text
    ])

# 应用函数到 DataFrame
columns = [
    '従業員数',
    '在宅可否', 
    '給与',
    '必要経験', 
    '仕事内容',  
    '配属部门', 
    '就業時間', 
    '選考内容', 
    '清洗后全文'
]

df[columns] = df['内容全文'].apply(extract_fields)

# 只有在确定不再需要原始列时才 drop
if '内容全文' in df.columns:
    df = df.drop(columns = ['内容全文'])



In [58]:
# 岗位类型，One-hot聚类

# ===== 1. 关键词词库=====
ROLE_DICT = {
    "DA": [
        "数据アナリスト", "データ分析", "データ集計", "マーケティングデータ",
        "ビジネスアナリスト", "Business Analyst", "データ解析", "インサイト分析",
        "顧客データ", "アナリティクス", "データ運用" 
    ],
    "DS": [
        "データサイエンティスト", "数理", "AI", "機械学習", 
        "Machine Learning", "LLM", "ゲノム解析", "画像解析", 
        "動作解析", "深層学習", "統計" 
    ],
    "DE": [
        "データエンジニア", "基盤", "構築", "DWH", "ETL", "Databricks",
        "データプラットフォーム", "Data Platform", "データ基盤", 
        "パイプライン", "アナリティクスエンジニア" 
    ],
    "MLE": [
        "機械学習エンジニア", "MLエンジニア", "AIエンジニア", 
        "ML Engineer", "Software Engineer(ML)" 
    ],
    "BI": [
        "BI", "ダッシュボード", "可視化", "Tableau", "Looker", 
        "PowerBI", "レポート作成" 
    ],
    "CONSULTANT": [
        "コンサル", "利活用", "データ活用", "戦略", "DX推進", 
        "マーケティング戦略", "企画", "調査", "リサーチ", "Research",
        "マーケター", "マーケティング担当", "市場開発", "Promotion",
        "プロダクトマネージャー", "事業開発", "PMM", "BizOps", "RevOps" 
    ]
}

# ===== 2. 优先级（保持不变，确保交叉学科时的准确性）=====
PRIORITY = ["DS", "MLE", "DE", "DA", "BI", "CONSULTANT"]

# ===== 3. 分类函数 (优化了大小写匹配) =====
def classify_role(title):
    if not isinstance(title, str):
        return "OTHER"
    
    # 统一转大写，处理如 "bi" 和 "BI" 的匹配问题
    title_upper = title.upper()
    matched_roles = []

    for role, keywords in ROLE_DICT.items():
        for kw in keywords:
            if kw.upper() in title_upper:
                matched_roles.append(role)
                break

    if not matched_roles:
        return "OTHER"

    # 按优先级返回最高的一个
    for p in PRIORITY:
        if p in matched_roles:
            return p

# ===== 4. 应用与验证 =====
df["role_type"] = df["Position"].apply(classify_role)

print("\n=== 岗位分布 ===")
print(df["role_type"].value_counts())


=== 岗位分布 ===
role_type
OTHER         133
CONSULTANT    123
DS            111
DE             39
DA             32
BI              5
Name: count, dtype: int64


In [59]:
# 在宅情况，One-hot聚类

# ===== 1. 关键词配置（保持你的逻辑，增强覆盖面）=====
REMOTE_KW = ["リモート", "在宅", "テレワーク", "フルリモート"]
ALLOW_KW = ["可", "可能", "OK", "利用", "対応", "推奨", "導入", "活用", "併用"]
NEGATIVE_REMOTE = ["不可", "実施しておりません", "対象外", "できません"]
ONSITE_KW = ["出社", "常駐", "対面"]

# ===== 2. 预处理函数 =====
def normalize_text(text):
    if not isinstance(text, str): return ""
    # 显式处理代码1生成的“无匹配”
    if text == "无匹配": return "NONE"
    # 去掉空格和换行，但保留 || 作为逻辑分隔（或者直接去掉也可）
    text = re.sub(r"\s+", "", text)
    return text

# ===== 3. 分类函数（保持判断逻辑，优化匹配深度）=====
def classify_remote(text):
    text = normalize_text(text)
    
    if text == "NONE" or text == "":
        return "UNKNOWN", -1

    # ---- ① 否定优先（针对远程词的否定） ----
    # 捕捉类似 “在宅不可”、“リモートワークは実施しておりません”
    for r_kw in REMOTE_KW:
        if r_kw in text:
            for n_kw in NEGATIVE_REMOTE:
                if n_kw in text and text.find(r_kw) < text.find(n_kw) < text.find(r_kw) + 10:
                    return "ONSITE", 0

    # ---- ② 强判定：组合命中（你的核心逻辑） ----
    for r_kw in REMOTE_KW:
        if r_kw in text:
            for a_kw in ALLOW_KW:
                if a_kw in text:
                    return "REMOTE", 1

    # ---- ③ 弱判定：补充命中（针对“一部従業員利用可”或“フルリモート”等固定搭配） ----
    # 很多文本里“可”在括号里，上面的逻辑有时会漏掉，这里做一层兜底
    if any(kw in text for kw in ["フルリモート", "リモートワーク制度", "在宅勤務制度"]):
        return "REMOTE", 1

    # ---- ④ 明确 onsite ----
    # 如果前面没匹配到远程许可，且出现了出社关键词
    for kw in ONSITE_KW:
        if kw in text:
            return "ONSITE", 0

    # ---- ⑤ 最后的兜底：含有远程词但无明确许可/否定 ----
    # 只要出现了在宅相关的词，在招聘语境下大概率是支持的，减少 UNKNOWN
    if any(kw in text for kw in REMOTE_KW):
        return "REMOTE", 1

    return "UNKNOWN", -1

# ===== 4. 应用 =====
# 假设 df 是你的原始 DataFrame
res = df["在宅可否"].apply(lambda x: pd.Series(classify_remote(x)))
df["remote_flag"] = res[0]
df["remote_score"] = res[1]

# ===== 5. 分布 =====
print("\n=== remote_flag分布 ===")
print(df["remote_flag"].value_counts())


=== remote_flag分布 ===
remote_flag
REMOTE     325
UNKNOWN    102
ONSITE      16
Name: count, dtype: int64


In [ ]:
# 年收进一步分类
def extract_salary(salary_text):
    if pd.isna(salary_text) or salary_text == "":
        return "UNKNOWN", "UNKNOWN", "UNKNOWN"
    
    # 统一转为字符串并去掉空格
    text = str(salary_text).replace(" ", "").replace(",", "")
    
    # 正则表达式：匹配 500万円～900万円 这种格式
    # 允许匹配：500万, 500万円, 500~900, 500-900 等变体
    pattern = r'(\d{3,4})(?:万円|万)?(?:～|-|~)(\d{3,4})(?:万円|万)?'
    match = re.search(pattern, text)
    
    if match:
        try:
            s_min = int(match.group(1))
            s_max = int(match.group(2))
            s_avg = (s_min + s_max) / 2
            return s_min, s_max, s_avg
        except ValueError:
            return "UNKNOWN", "UNKNOWN", "UNKNOWN"
    
    # 兜底逻辑：如果只有单值（如 500万円～）
    single_pattern = r'(\d{3,4})万円'
    single_match = re.search(single_pattern, text)
    if single_match:
        try:
            s_min = int(single_match.group(1))
            return s_min, "UNKNOWN", "UNKNOWN"
        except ValueError:
            pass

    return "UNKNOWN", "UNKNOWN", "UNKNOWN"

# ===== 应用清洗 =====
# 假设你的 DataFrame 叫 df，“给与”列是年收所在列
salary_data = df["給与"].apply(extract_salary)

# 将结果展开到三列
df[['salary_min', 'salary_max', 'salary_avg']] = pd.DataFrame(salary_data.tolist(), index=df.index)

# ===== 打印结果查看 =====
print(df[['給与', 'salary_min', 'salary_max', 'salary_avg']].head())

                                                  給与  salary_min salary_max  \
0  想定年収 500万円～900万円 賃金 月給制 月給 371,000円～667,000円 月...         500        900   
1  想定年収 500万円～900万円 賃金 月給制 月給 371,000円～667,000円 月...         500        900   
2  想定年収 500万円～800万円 賃金 月給制 月給 416,667円～666,667円 月...         500        800   
3  想定年収 600万円～1,000万円 賃金 月給制 月給 500,000円～833,000円...         600       1000   
4  想定年収 757万円～930万円 賃金 月給制 月給 631,000円～775,000円 月...         757        930   

  salary_avg  
0      700.0  
1      700.0  
2      650.0  
3      800.0  
4      843.5  


In [64]:

def extract_experience_years_v2(text):
    if pd.isna(text) or text == "" or text == "无匹配":
        return 0
    
    text = str(text)
    
    # --- 1. 优先提取具体年数 ---
    # 匹配：3年以上、5年以上の実務経験、经验2年 等
    year_match = re.search(r'(\d{1,2})\s*年以上?|経験(\d{1,2})\s*年', text)
    if year_match:
        # 取出匹配到的第一个非空数字组
        val = year_match.group(1) if year_match.group(1) else year_match.group(2)
        return int(val)
    
    # 匹配：社会人年次2年目～5年目 (取最小值)
    genji_match = re.search(r'社会人年次(\d{1,2})年目', text)
    if genji_match:
        return int(genji_match.group(1))

    # --- 2. 逻辑调整：必须有项目经验但未写年数的设为 1 ---
    # 定义表示“有经验”的关键词
    has_exp_keywords = [
        "経験", "実務", "開発経験", "運用経験", "構築経験", 
        "携わった", "職務での", "スキルをお持ちの方"
    ]
    
    if any(kw in text for kw in has_exp_keywords):
        return 1

    # --- 3. 彻底没提到的设为 0 ---
    return 0

# ===== 应用清洗 =====
df["experience_years"] = df["必要経験"].apply(extract_experience_years_v2)

# ===== 验证结果展示 =====
# 选取几个典型案例看看效果
sample_check = df[df["必要経験"].str.contains("経験", na=False)].copy()
print("\n=== 经验年数提取结果示例 ===")
print(sample_check[["必要経験", "experience_years"]].head(10))

# 查看分布
print("\n=== 经验年数分布 ===")
print(df["experience_years"].value_counts().sort_index())


=== 经验年数提取结果示例 ===
                                                必要経験  experience_years
0  【必須】■Pythonを用いた機械学習・データ分析の実務経験 ■AI・データ分析エンジニアと...                 2
1  【必須】■システム開発エンジニアとしての実務経験3年以上 ■フロントエンド（React等）o...                 3
2  【必須経験】社会人経験3年以上■営業経験／顧客折衝経験（営業・CS・リーダー職など）■DX/...                 3
3  【必須】■電力業界での実務経験（業種・職種は問わない） 【歓迎】■需給管理・電力トレーディン...                 1
4  【必須】■高度なSQLスキル（クエリチューニングや複雑なロジック構築ができる）■gitコマン...                 1
5  【必須】 ・保険代理店や顧客対応を含む業務経験（2年以上） 上記＋下記いずれかの経験 ・保険...                 2
6  【必須】・保険業界の本社/企画部門における2年以上の業務経験 ・生命保険／損害保険／少額短期...                 2
7  【いずれか必須】 ■事業開発のご経験（スタートアップ、ベンチャー、メガベンチャー、または大手...                 1
8  【いずれか必須】 ■事業開発のご経験（スタートアップ、ベンチャー、メガベンチャー、または大手...                 1
9  【必須】 ・金融会社（銀行・証券等）でのFinTech系サービス企画・開発経験が1年以上ある...                 1

=== 经验年数分布 ===
experience_years
0      17
1     278
2      44
3      66
5      30
6       1
7       4
8       1
10      2
Name: count, dtype: int64


In [ ]:


def classify_level(row):
    # 获取 Position 和之前清洗出的经验年数
    position = str(row['Position']).upper()
    exp = row['experience_years']
    
    # ===== 1. MANAGER (管理/领导层) =====
    # 关键词包含：经理、组长、PM、负责人、管理监督者等
    manager_keywords = [
        "マネージャー", "MANAGER", "リーダー", "LEADER", "PM", "PMO", "PL", "CMO",
        "室長", "課長", "部長", "責任者", "幹部", "管理监督者", "管理職", "パートナー"
    ]
    if any(kw in position for kw in manager_keywords):
        return "manager"

    # ===== 2. SENIOR (资深/专家/上流工程) =====
    # 关键词包含：资深、专家、首席、架构师、设计、战略、以及 3 年以上经验
    senior_keywords = [
        "リード", "LEAD", "シニア", "SENIOR", "エキスパート", "EXPERT", 
        "設計", "上流", "アーキテクト", "ARCHITECT", "戦略", "エバンジェリスト",
        "プロ採用"
    ]
    if any(kw in position for kw in senior_keywords) or exp >= 3:
        return "senior"

    # ===== 3. JUNIOR (初级/潜力/未开发) =====
    # 关键词包含：未经验、潜力、研修、第二新卒、或是经验为 0 且不含管理词
    junior_keywords = [
        "未経験", "ポテンシャル", "研修", "第二新卒", "アシスタント", 
        "新卒", "オープンポジション", "OPEN POSITION", "メンバークラス"
    ]
    if any(kw in position for kw in junior_keywords) or exp == 0:
        return "junior"

    # ===== 4. MID (中坚/中级) =====
    # 默认分类：有一定实务经验（1-3年），且职位没有明确标注为管理或资深
    return "mid"

# ===== 应用函数 =====
df['level'] = df.apply(classify_level, axis=1)

# ===== 打印分布情况 =====
print("\n=== Level 分布统计 ===")
print(df["level"].value_counts())



=== Level 分布统计 ===
level
mid        190
senior     119
manager     80
junior      54
Name: count, dtype: int64

=== 典型样本分类结果 ===
Empty DataFrame
Columns: [Position, experience_years, level]
Index: []


In [69]:
df.to_csv('test.csv', index=False, encoding='utf-8-sig')